# LangGraph — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/langgraph.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/langgraph.ipynb)

Companion to: [LangGraph](https://anacodicai.org/learn/ai-agents/6-2/langgraph)


## What you'll build

This is the **real `langgraph` library** — not a simulation. Same trip-planning
assistant as the article's real-world example: a group thread where the AI
decides whether to act on a suggestion. We build a small state graph with:

1. A node that decides on an action and how confident it is in it.
2. A **conditional edge** that routes to a human-approval pause when
   confidence is low, and straight to persistence when it's high.
3. A **checkpointed interrupt/resume**: the graph genuinely pauses execution,
   and a *separate* `.invoke()` call resumes it exactly where it left off —
   the same mechanism a real auto-booking assistant needs before it commits
   money on your behalf.

**No API key needed** — the "LLM" is a plain Python function standing in for
one, so the graph mechanics are 100% real while the "AI" step is a mock.
`pip install langgraph` if running locally; Colab needs it too (see Setup).

### Setup

In [1]:
# %pip install -q langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

print("real langgraph imported")

real langgraph imported


## Step 1 — the state

A `TypedDict` is the graph's shared memory — every node reads from it and
returns a partial update that gets merged in.

In [2]:
class State(TypedDict, total=False):
    request: str
    confidence: float
    action: str
    override: str

print("State schema defined")

State schema defined


## Step 2 — the nodes

`plan_node` calls our mock "LLM" (swap in a real API call here and nothing
else about the graph changes). `human_review_node` is where the real
LangGraph mechanism lives: `interrupt(...)` genuinely suspends execution.
`persist_node` uses the human's override if one exists, otherwise the
assistant's own planned action.

In [3]:
def mock_plan_trip_action(request: str) -> dict:
    # stand-in for a real LLM call -- swap this one function and the
    # rest of the graph is unchanged
    if "everyone confirmed" in request.lower():
        return {"action": "book flight to Barcelona", "confidence": 0.95}
    return {"action": "hold: suggest flight to Barcelona", "confidence": 0.4}

def plan_node(state: State) -> State:
    result = mock_plan_trip_action(state["request"])
    return {"action": result["action"], "confidence": result["confidence"]}

def route_after_plan(state: State) -> str:
    return "human_review" if state["confidence"] < 0.75 else "persist"

def human_review_node(state: State) -> State:
    decision = interrupt({"message": "low confidence, please review", "action": state["action"]})
    return {"override": decision}

def persist_node(state: State) -> State:
    final = state.get("override") or state["action"]
    return {"action": final}

print("nodes defined")

nodes defined


## Step 3 — wire the graph

**Try it:** change the confidence threshold in `route_after_plan` from `0.75`
to `0.3` and re-run everything below — the "everyone confirmed" case should
stop pausing entirely.

In [4]:
g = StateGraph(State)
g.add_node("plan", plan_node)
g.add_node("human_review", human_review_node)
g.add_node("persist", persist_node)

g.add_edge(START, "plan")
g.add_conditional_edges("plan", route_after_plan,
                         {"human_review": "human_review", "persist": "persist"})
g.add_edge("human_review", "persist")
g.add_edge("persist", END)

checkpointer = InMemorySaver()
graph = g.compile(checkpointer=checkpointer)
print("graph compiled with a real checkpointer")

graph compiled with a real checkpointer


## Step 4 — a confident case: no pause at all

Everyone in the thread already agreed on dates, so the assistant is
confident enough to just book it.

In [5]:
confident_config = {"configurable": {"thread_id": "demo-confident"}}
result_confident = graph.invoke(
    {"request": "Everyone confirmed dates for the Barcelona trip"}, config=confident_config
)
print("confident case result:", result_confident)
assert "__interrupt__" not in result_confident
print("assert passed: high-confidence case never paused")

confident case result: {'request': 'Everyone confirmed dates for the Barcelona trip', 'confidence': 0.95, 'action': 'book flight to Barcelona'}
assert passed: high-confidence case never paused


## Step 5 — a low-confidence case: it genuinely pauses

Just a maybe from one person isn't enough to auto-book on. Note the
`__interrupt__` key in the result — that's LangGraph telling us execution
stopped mid-graph and is waiting for a human.

In [6]:
low_conf_config = {"configurable": {"thread_id": "demo-low-confidence"}}
result_paused = graph.invoke(
    {"request": "Someone mentioned maybe Barcelona for the trip"}, config=low_conf_config
)
print("paused result:", result_paused)
assert "__interrupt__" in result_paused
print("assert passed: low-confidence case genuinely paused for human review")

paused result: {'request': 'Someone mentioned maybe Barcelona for the trip', 'confidence': 0.4, 'action': 'hold: suggest flight to Barcelona', '__interrupt__': [Interrupt(value={'message': 'low confidence, please review', 'action': 'hold: suggest flight to Barcelona'}, id='6ed5816f1012d2c13225aff5cc4950c4')]}
assert passed: low-confidence case genuinely paused for human review


## Step 6 — resume it, in a SEPARATE call, with a human decision

This is the part that isn't a simulation: `graph.invoke(Command(resume=...))`
picks the run back up from its checkpoint — the same `thread_id`, a brand new
Python call, and the graph continues exactly where `human_review_node` left
off, with `interrupt(...)`'s return value now set to whatever the trip
organizer decided.

In [7]:
resumed = graph.invoke(
    Command(resume="Approved: book flight to Barcelona for the confirmed dates"),
    config=low_conf_config,
)
print("after resume:", resumed)

assert resumed["action"] == "Approved: book flight to Barcelona for the confirmed dates"
assert resumed["override"] == "Approved: book flight to Barcelona for the confirmed dates"
print("\nassert passed: the resumed run applied the human's approval correctly")

after resume: {'request': 'Someone mentioned maybe Barcelona for the trip', 'confidence': 0.4, 'action': 'Approved: book flight to Barcelona for the confirmed dates', 'override': 'Approved: book flight to Barcelona for the confirmed dates'}

assert passed: the resumed run applied the human's approval correctly


## Try it yourself

- Add a THIRD node between `plan` and `persist` for, say, checking the
  group's past preferences before proposing an action — real multi-agent
  planning pipelines often also have a preference-memory stage, a
  consistency check across specialists' suggestions, and a human-approval
  gate, just with more steps than this toy example.
- Swap `mock_plan_trip_action` for a real API call (OpenAI, Anthropic, Groq)
  — the graph wiring, the conditional routing, and the interrupt/resume
  mechanics don't change at all. That's the actual point of LangGraph: the
  orchestration layer is independent of which model is doing the "thinking"
  inside a node.
- Try calling `graph.invoke(Command(resume=...))` on a `thread_id` that was
  never paused — see what error you get, and think about why a real system
  needs to check `__interrupt__` before ever calling resume.